# LowResPT — Patch Reconstruction

Reconstruct full spectra from the model's overlapping patch tokens, under **two
protocols on the same frozen checkpoint**:

- **Section A · Full-visible** — every token is visible; a single forward pass
  reconstructs all patches. This is the regime `val_loss` reports, but it is an
  *untrained* regime (the model never sees fully-unmasked inputs at train time),
  so its reconstructions can look deceptively flat / wrong near emission peaks.
- **Section B · Masked (MAE-consistent)** — for each token a leak-free `BLOCK_K`
  block centred on it is masked, the model predicts the hidden token from
  context, and we keep that prediction. This matches the masked-autoencoding
  objective the model was trained on and is the **trustworthy** readout.

Overlapping pixels are averaged across the patches that cover them. The dataset
is always observed-frame; analysis cells divide by `(1+z)` inline to plot
rest-frame.

## Config

In [ ]:
import sys, math, torch, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '.')

from model.low_res_pt import LowResPT
from data.datamodule import LowResDataModule
from data.dataset import LowResDataset

# Single checkpoint — Section A and Section B are two reconstruction protocols
# evaluated on THIS model. Point it anywhere you like.
from pathlib import Path
NAME, VERSION = "low_res_pt_1_2_micron_noz_cut", 0   # run name / region, and version_<N>
CKPT = min(Path("outputs").glob(f"{NAME}/version_{VERSION}/checkpoints/*.ckpt"),
           key=lambda p: float(p.stem.split("val_hid_loss=")[-1]))   # best val_hid_loss
print("CKPT:", CKPT)
FITS   = "/home/yacheng/ssl_outthere/data/spectrum/DJA_spectra_v4.5.fits"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

USE_JANSKY       = False   # dataset flux units (must match how the model was trained)
VAL_SN_THRESHOLD = 0     # post-split S/N cut: restrict val set to high-S/N spectra

# NOTE: this is an absolute count of positions per end, applied to BOTH the pixel
# mask (length L_used) AND the token mask (length N). 
EDGE_TRIM = 2

# Masked reconstruction (Section B): how many CONSECUTIVE tokens to mask around
# each target t.
# K=1 : leave-one-out. Leak-free iff S == P (yaml S=4, P=4 -> no overlap ->
#       1 token mask hides its 4 pixels completely).
# K=2 : 2-token block. Hides up to 2K=4 pixels truly when stride < P.
# K=3 : 3-token block (designed for P=4/S=2 overlap). Over-hides on S=4.
# Match your model's stride: S==P -> use 1; S<P -> use 2 or 3.
BLOCK_K = 3

## Load frozen model

In [ ]:
model = LowResPT.load_from_checkpoint(CKPT, map_location=DEVICE).eval()
print(model.hparams)

wl_max = model.hparams['wl_ref_max']
wl_min = model.hparams['wl_ref_min']
print('model wavelength range:', wl_min, '-', wl_max)

## Validation split (shared)

In [ ]:
from torch.utils.data import Subset

# Split EXACTLY as in training so the val split matches the training run's val
# split (random_split seed=42, train_val_split=0.9 -- both hardcoded).

#in the following both dataset and datamodule are used.
ds = LowResDataset(fits_path=FITS, 
                        min_sn50=1.0, min_redshift=0.0, use_jansky=USE_JANSKY,
                        wl_ref_min=wl_min, wl_ref_max=wl_max, frac_valid_pix=0.5)

dm = LowResDataModule(fits_path=FITS, batch_size=256, num_workers=0,
                        min_sn50=1.0, min_redshift=0.0, use_jansky=USE_JANSKY,
                        wl_ref_min=wl_min, wl_ref_max=wl_max, frac_valid_pix=0.5)
dm.setup()

# Post-split S/N threshold: restrict the val set to high-S/N spectra. Applied
# AFTER the split, so it only sub-selects val and never perturbs the partition.
# base.sn50 is dataset-order (aligned with base); val_local indexes into base.
base      = dm.val_dataset.dataset                  # underlying LowResDataset
val_local = np.asarray(dm.val_dataset.indices)      # local idx into base dataset
keep      = [int(i) for i in val_local if base.sn50[int(i)] >= VAL_SN_THRESHOLD]
dm.val_dataset = Subset(base, keep)                 # all downstream cells use this
print(f"val split: {len(val_local)} spectra  ->  {len(keep)} with sn50 >= {VAL_SN_THRESHOLD}")

batch = next(iter(dm.val_dataloader()))
print(f"batch flux shape: {batch['flux'].shape},  device: {DEVICE}")

## Helpers — reconstruction & analysis

A single `reconstruct(model, batch, masked=, block_k=)` covers both protocols
and returns an identical dict, so every analysis function below works unchanged
for Section A and Section B. The analyses are written as functions that take a
`recon_fn` closure (`lambda b: reconstruct(model, b, masked=...)`), so each
section just calls the same suite.

In [ ]:
@torch.no_grad()
def reconstruct(model, batch, masked=False, block_k=1):
    # Reconstruct patch tokens, overlap-averaged into a pixel spectrum.
    #   masked=False : single forward pass, all tokens visible (full-visible regime).
    #   masked=True  : iterative reconstruction, matches the training objective
    #                  for each token t, mask a leak-free block_k-token block centred
    #                  on t and keep that pass's prediction for t (MAE-consistent).
    #                  Masked tokens' patches AND patch_stats are zeroed (matches the
    #                  training leak-free convention). Cost: N forward passes/batch.
    # Returns the same dict shape in both modes.
    
    flux  = batch["flux"].to(DEVICE)
    wave  = batch["wavelength"].to(DEVICE)
    vmask = batch["valid_mask"].to(DEVICE)
    z     = batch["redshift"]                              # (B,) on CPU

    # Core encode/decode (both protocols) lives in model.reconstruct().
    R = model.reconstruct(flux, wave, vmask, masked=masked, block_k=block_k)
    recon_pat   = R["recon_patches"]                       # (B, N, P) predicted
    patches     = R["target"]                              # (B, N, P) target
    flux_norm   = R["flux_norm"]                           # (B, L)
    token_vmask = R["token_valid_mask"]                    # (B, N) model attn token mask
    valid_patches = R["valid_patches"]                     # (B, N, P) per-pixel validity
    L_used      = R["L_used"]
    mean, std   = R["stats"][:, :1], R["stats"][:, 1:]     # (B, 1) each
    B, N, P     = patches.shape
    S           = model.hparams.stride

    # ── Overlap averaging, FULLY-valid tokens only: patch t covers pixels [t*S, t*S+P).
    # A token contributes ONLY if it has zero bad pixels (token_full_valid); any
    # token containing a bad pixel is dropped from the reconstruction entirely.
    # Since a fully-valid token covers only valid pixels, every kept pixel is itself
    # valid -> recon_mask is automatically a SUBSET of vmask, so this is the single
    # mask downstream needs. A pixel is kept iff >=1 fully-valid token covers it; a
    # pixel covered only by bad-pixel-containing tokens is dropped (its neighbours
    # inside a fully-valid token are still reconstructed, just without that token).
    token_full_valid = valid_patches.all(dim=-1)           # (B, N) zero bad pixels
    recon_pix = torch.zeros(B, L_used, device=DEVICE)
    count     = torch.zeros(B, L_used, device=DEVICE)
    for t in range(N):
        s = t * S
        w = token_full_valid[:, t].to(recon_pat.dtype).unsqueeze(-1)  # (B,1) 1=fully-valid
        recon_pix[:, s:s+P] += recon_pat[:, t, :] * w
        count[:, s:s+P]     += w
    recon_mask = count > 0                                  # (B, L_used) bool; subset of vmask
    recon_pix = recon_pix / count.clamp(min=1)             # norm space; 0 where count=0

    #a new dict for output
    return dict(
        flux       = flux[:, :L_used].cpu(),
        wave       = wave[:, :L_used].cpu(),
        vmask      = vmask[:, :L_used].cpu(),
        recon_mask = recon_mask.cpu(),      # (B, L_used) >=1 valid token covers pixel
        flux_norm  = flux_norm[:, :L_used].cpu(),
        recon_norm = recon_pix.cpu(),
        recon_flux = (recon_pix * std + mean).cpu(),       # original flux units
        patches    = patches.cpu(),         # (B, N, P) target patches
        recon_pat  = recon_pat.cpu(),       # (B, N, P) predicted patches
        token_vmask= token_vmask.cpu(),     # (B, N)
        count      = count.cpu(),           # (B, L_used) overlap count per pixel
        mean=mean.cpu(), std=std.cpu(),
        redshift   = z.cpu() if isinstance(z, torch.Tensor) else torch.as_tensor(z),
        N=N, P=P, S=S, L_used=L_used,
    )

In [ ]:
# Rest-frame emission lines (µm), sorted by wavelength.
EMISSION_LINES = {
    r'Ly$\alpha$':          0.1216,
    '[OII]':                0.3727,
    r'H$\beta$':            0.4861,
    '[OIII]':               0.5007,
    r'H$\alpha$':           0.6563,
    '[SII]':                0.6724,
    r'Pa$\zeta$':           0.9229,
    r'Pa$\epsilon$+[SIII]': 0.9539,
    r'Pa$\delta$':          1.0049,
    r'Pa$\gamma$':          1.0938,
}

def _annotate_elines(ax, w_min, w_max, fontsize=8):
    # Draw rest-frame emission-line markers with staggered labels.
    visible = [(name, wl) for name, wl in EMISSION_LINES.items()
               if w_min <= wl <= w_max]
    if not visible:
        return
    visible.sort(key=lambda x: x[1])
    ylo, yhi = ax.get_ylim()
    yspan    = yhi - ylo
    y_levels = [yhi - 0.04 * yspan, yhi - 0.20 * yspan]
    for i, (name, wl) in enumerate(visible):
        ax.axvline(wl, color='#999999', ls='--', lw=1.0, alpha=0.8)
        ax.text(wl, y_levels[i % 2], name, ha='center', va='top',
                fontsize=fontsize, color='#333333', fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.75, pad=1.5, edgecolor='none'))


def recon_summary(R, tag=""):
    # Per-token / per-sample MSE stats + overlap-count diagnostic.
    tok_mse = ((R["recon_pat"] - R["patches"]) ** 2).mean(-1)          # (B, N)
    pmask   = R["recon_mask"].float()                     # fully-valid-token reconstruction (subset of vmask)
    pix_err = ((R["recon_norm"] - R["flux_norm"]) ** 2) * pmask
    pix_mse = pix_err.sum(-1) / pmask.sum(-1).clamp(min=1)

    print(f"[{tag}]  Pixel MSE (norm)  mean={pix_mse.mean():.4f}  std={pix_mse.std():.4f}  median={pix_mse.median():.4f}")
    print(f"[{tag}]  Token MSE (norm)  mean={tok_mse[R['token_vmask']].mean():.4f}  std={tok_mse[R['token_vmask']].std():.4f}")
    print(f"[{tag}]  Overlap count     min={R['count'].min():.0f}  max={R['count'].max():.0f}"
          f"  (P={R['P']}, S={R['S']} -> expect max {R['P']//R['S']})")

    fig, axes = plt.subplots(1, 3, figsize=(13, 3))
    axes[0].hist(pix_mse.numpy(), bins=40, edgecolor="none")
    axes[0].set(title="Per-sample pixel MSE (norm)", xlabel="MSE", ylabel="count")
    axes[1].hist(tok_mse[R["token_vmask"]].numpy(), bins=40, edgecolor="none")
    axes[1].set(title="Per-token MSE (norm)", xlabel="MSE", ylabel="count")
    axes[1].set_yscale("log")
    axes[2].plot(R["count"][0].numpy(), lw=1.5)
    axes[2].set(title="Pixel overlap count -- sample 0", xlabel="pixel index",
                ylabel="n patches covering pixel")
    fig.suptitle(f"Reconstruction stats -- {tag}", fontweight="bold")
    plt.tight_layout(); plt.show()
    return tok_mse


def gallery(R, tag="", n_gallery=128, space='normalized', max_cols=4,
            panel_w=3.5, panel_h=2.5, seed=0):
    # Grid of original vs reconstructed spectra (rest-frame).
    rng    = np.random.default_rng(seed)
    N_tot  = R["flux"].shape[0]
    n_plot = min(n_gallery, N_tot)
    idxs   = rng.choice(N_tot, size=n_plot, replace=False)
    n_cols = min(max_cols, n_plot)
    n_rows = math.ceil(n_plot / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(panel_w * n_cols, panel_h * n_rows),
                             squeeze=False)
    flat = axes.flatten()
    legend_done = False          # legend goes on the first VISIBLE panel (panel 0 may be masked out)
    for ai, idx in enumerate(idxs):
        ax = flat[ai]
        m  = R["recon_mask"][idx].clone()
        L_used = m.shape[0]
        if EDGE_TRIM > 0:
            m[:EDGE_TRIM] = False
            m[L_used - EDGE_TRIM:] = False
        if m.sum() == 0:
            ax.set_visible(False); continue
        # Observed-frame in; divide by (1+z) to plot rest-frame.
        w  = (R["wave"][idx] / (1.0 + R["redshift"][idx].item()))[m].numpy()
        si = np.argsort(w); w = w[si]
        if space == 'normalized':
            orig  = R["flux_norm"][idx][m].numpy()[si]
            recon = R["recon_norm"][idx][m].numpy()[si]
            ylab  = "Norm. flux"
        else:
            mn, sd = R["mean"][idx].item(), R["std"][idx].item()
            orig  = R["flux"][idx][m].numpy()[si]
            recon = R["recon_norm"][idx][m].numpy()[si] * sd + mn
            ylab  = "Flux (uJy)"
        mse = float(np.mean((orig - recon) ** 2))
        ax.plot(w, orig,  color='steelblue', lw=1.0, alpha=0.85, label='Orig')
        ax.plot(w, recon, color='tomato',    lw=1.0, alpha=0.85, ls='--', label='Recon')
        _annotate_elines(ax, w[0], w[-1], fontsize=6)   # rest-frame emission-line markers
        ax.set_title(f'#{idx}  MSE={mse:.3f}', fontsize=7)
        ax.set_xlabel('λ rest (µm)', fontsize=6)
        ax.set_ylabel(ylab, fontsize=6)
        ax.tick_params(labelsize=5); ax.grid(alpha=0.25)
        if not legend_done:
            ax.legend(fontsize=5, loc='upper right')
            legend_done = True
    for ai in range(n_plot, len(flat)):
        flat[ai].set_visible(False)
    stag = 'Normalized' if space == 'normalized' else 'Physical (uJy)'
    plt.suptitle(f'Patch Reconstruction ({tag}) -- {stag}  '
                 f'[n={n_plot}, P={R["P"]}, S={R["S"]}, edge_trim={EDGE_TRIM}]',
                 fontsize=10, fontweight='bold')
    plt.tight_layout(); plt.show()


def per_token_bar(R, tok_mse, i=20, tag=""):
    # Per-token reconstruction error for a single sample.
    P, S = R["P"], R["S"]
    w_full = R["wave"][i] / (1.0 + R["redshift"][i].item())
    tok_w  = w_full[:R["L_used"]].unfold(0, P, S).mean(-1).numpy()
    tok_e  = tok_mse[i].numpy()
    tv     = R["token_vmask"][i].clone().numpy()
    N_tok  = len(tv)
    if EDGE_TRIM > 0:
        tv[:EDGE_TRIM] = False
        tv[N_tok - EDGE_TRIM:] = False
    fig, ax = plt.subplots(figsize=(10, 2.5))
    ax.bar(tok_w[tv], tok_e[tv], width=0.012, alpha=0.8, color="steelblue")
    ax.set_xlabel("rest wavelength (μm)")
    ax.set_ylabel("token MSE (norm)")
    ax.set_title(f"Per-token reconstruction error ({tag}) -- sample {i}  "
                 f"(P={P}, S={S}, N={R['N']} tokens, edge_trim={EDGE_TRIM})")
    ax.set_yscale('log')
    plt.tight_layout(); plt.show()

In [ ]:
def mse_vs_wavelength(recon_fn, tag="", n_lambda_bins=40, n_mse_bins=60,
                      log_mse_min=-4.0, log_mse_max=1.0, cmap='viridis'):
    # 2D histogram of reconstruction MSE vs rest-frame wavelength (all val).
    all_wave, all_err2 = [], []
    for b in dm.val_dataloader():
        Rb = recon_fn(b)
        m  = Rb["recon_mask"].clone()
        L_used = m.shape[1]
        if EDGE_TRIM > 0:
            m[:, :EDGE_TRIM] = False
            m[:, L_used - EDGE_TRIM:] = False
        w  = Rb["wave"] / (1.0 + Rb["redshift"].unsqueeze(-1))
        e2 = (Rb["recon_norm"] - Rb["flux_norm"]) ** 2
        all_wave.append(w[m].numpy()); all_err2.append(e2[m].numpy())
    wave_all = np.concatenate(all_wave); err2_all = np.concatenate(all_err2)
    print(f"[{tag}]  Total valid pixels : {len(wave_all):,}  (edge_trim={EDGE_TRIM})")
    print(f"[{tag}]  Mean pixel MSE     : {err2_all.mean():.4f}  +-  {err2_all.std():.4f}")

    valid    = err2_all > 0
    log_err2 = np.log10(err2_all[valid]); wave_v = wave_all[valid]
    log_bins = np.linspace(log_mse_min, log_mse_max, n_mse_bins + 1)
    lam_bins = np.linspace(wave_v.min(), wave_v.max(), n_lambda_bins + 1)
    h, xedge, yedge = np.histogram2d(wave_v, log_err2, bins=[lam_bins, log_bins])

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    yticks = np.arange(log_mse_min, log_mse_max + 0.5, 1.0)
    ax = axes[0]
    im = ax.pcolormesh(xedge, yedge, h.T, cmap=cmap)
    fig.colorbar(im, ax=ax, label='pixel count')
    ax.set_yticks(yticks); ax.set_yticklabels([f'$10^{{{int(t)}}}$' for t in yticks])
    ax.set_xlabel('rest wavelength (µm)')
    ax.set_ylabel('pixel squared error (norm, log scale)')
    ax.set_title('2D histogram -- log(MSE) vs λ')

    bin_centers = 0.5 * (xedge[:-1] + xedge[1:])
    bin_idx = np.digitize(wave_v, xedge) - 1
    bin_idx = bin_idx.clip(0, n_lambda_bins - 1)
    median_log = np.array([np.median(log_err2[bin_idx == k]) if (bin_idx == k).any() else np.nan
                           for k in range(n_lambda_bins)])
    p84_log = np.array([np.percentile(log_err2[bin_idx == k], 84) if (bin_idx == k).any() else np.nan
                        for k in range(n_lambda_bins)])
    ax = axes[1]
    ax.plot(bin_centers, median_log, color='C0', lw=1.5, label='median')
    ax.fill_between(bin_centers, median_log, p84_log, alpha=0.25, color='C0', label='50–84th pct')
    ax.set_yticks(yticks); ax.set_yticklabels([f'$10^{{{int(t)}}}$' for t in yticks])
    ax.set_xlabel('rest wavelength (µm)')
    ax.set_ylabel('pixel squared error (norm, log scale)')
    ax.set_title('Median log(MSE) vs λ'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.suptitle(f'{tag} -- reconstruction error vs wavelength  '
                 f'(n={len(wave_v):,} valid pixels, edge_trim={EDGE_TRIM})',
                 fontsize=10, fontweight='bold')
    plt.tight_layout(); plt.show()


def recon_vs_orig(recon_fn, tag="", max_scatter_pts=8000):
    # Per-sample MSE histogram, orig-vs-recon scatter, residual distribution.
    mse_list, mae_list, orig_pool, recon_pool = [], [], [], []
    for b in dm.val_dataloader():
        Rb = recon_fn(b)
        m  = Rb["recon_mask"].bool()
        for i in range(Rb["flux_norm"].shape[0]):
            mi = m[i]
            if mi.sum() == 0:
                continue
            o = Rb["flux_norm"][i][mi].numpy()
            r = Rb["recon_norm"][i][mi].numpy()
            mse_list.append(float(np.mean((o - r) ** 2)))
            mae_list.append(float(np.mean(np.abs(o - r))))
            orig_pool.append(o); recon_pool.append(r)
    mse_arr  = np.array(mse_list); mae_arr = np.array(mae_list)
    orig_all = np.concatenate(orig_pool); recon_all = np.concatenate(recon_pool)
    resid    = recon_all - orig_all


    #rng_sc = np.random.default_rng(0)
    #if len(orig_all) > max_scatter_pts:
    #    sel = rng_sc.choice(len(orig_all), max_scatter_pts, replace=False)
    #    o_sc, r_sc = orig_all[sel], recon_all[sel]
    #else:
    o_sc, r_sc = orig_all, recon_all
    r_val = np.corrcoef(o_sc, r_sc)[0, 1]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ax = axes[0]
    ax.hist(mse_arr, bins=35, color='steelblue', alpha=0.8, edgecolor='black', lw=0.4)
    ax.axvline(mse_arr.mean(), color='red', ls='--', lw=1.5,
               label=f'mean={mse_arr.mean():.4f}  std={mse_arr.std():.4f}')
    ax.set_xlabel('Per-sample MSE (norm)'); ax.set_ylabel('Count')
    ax.set_title('MSE Distribution'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1]
    # 2D density (log-scaled counts) instead of a scatter: dense cores stay
    # readable when there are millions of pixels. Empty hexes left blank (mincnt=1).
    # Independent, percentile-clipped ranges per axis (x fits Original, y fits
    # Reconstructed) so a handful of extreme pixels don't stretch the view.
    PCT = (0.1, 99.9)
    xlo, xhi = np.percentile(o_sc, PCT)
    ylo, yhi = np.percentile(r_sc, PCT)
    hb = ax.hexbin(o_sc, r_sc, gridsize=90, bins='log', cmap='viridis',
                   extent=(xlo, xhi, ylo, yhi), mincnt=1)
    fig.colorbar(hb, ax=ax, label='log$_{10}$(pixel count)')
    d_lo, d_hi = max(xlo, ylo), min(xhi, yhi)          # 1:1 line over common range
    ax.plot([d_lo, d_hi], [d_lo, d_hi], 'w--', lw=1, label='1:1')
    ax.set_xlim(xlo, xhi); ax.set_ylim(ylo, yhi)
    ax.set_xlabel('Original (norm)'); ax.set_ylabel('Reconstructed (norm)')
    ax.set_title(f'2D density  r={r_val:.3f}  (n={len(o_sc):,})')
    ax.legend(fontsize=8, loc='upper left')

    ax = axes[2]
    ax.hist(resid, bins=50, color='mediumseagreen', alpha=0.8, edgecolor='black', lw=0.4)
    ax.axvline(0, color='k', ls='--', lw=1.0)
    ax.axvline(resid.mean(), color='red', ls='-', lw=1.5,
               label=f'mean={resid.mean():.4f}  std={resid.std():.4f}')
    ax.set_xlabel('Residual (Recon − Orig)'); ax.set_ylabel('Count')
    ax.set_title('Residual Distribution'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.suptitle(f'{tag} -- Reconstruction Statistics [Normalized]  n={len(mse_arr)}\n'
                 f'MSE={mse_arr.mean():.4f}±{mse_arr.std():.4f}   '
                 f'MAE={mae_arr.mean():.4f}±{mae_arr.std():.4f}',
                 fontsize=10, fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
def outlier_analysis(recon_fn, tag="", outlier_nsigma=5.0, n_gallery_out=128):
    # Flag pixels whose residual deviates > nsigma from the global mean, then plot
    # scatter / wavelength distribution / outlier-rate, plus a gallery of the
    # spectra with the most outlier pixels.
    records = []
    for b in dm.val_dataloader():
        Rb = recon_fn(b)
        m  = Rb["recon_mask"].bool().clone()
        L_used = m.shape[1]
        if EDGE_TRIM > 0:
            m[:, :EDGE_TRIM] = False
            m[:, L_used - EDGE_TRIM:] = False
        w_rest = Rb["wave"] / (1.0 + Rb["redshift"].unsqueeze(-1))
        for i in range(Rb["flux_norm"].shape[0]):
            mi = m[i]
            if mi.sum() == 0:
                continue
            records.append((Rb["flux_norm"][i][mi].numpy(),
                            Rb["recon_norm"][i][mi].numpy(),
                            w_rest[i][mi].numpy()))
    orig_pix  = np.concatenate([x[0] for x in records])
    recon_pix = np.concatenate([x[1] for x in records])
    wave_pix  = np.concatenate([x[2] for x in records])
    resid_pix = recon_pix - orig_pix

    r_mean, r_std = resid_pix.mean(), resid_pix.std()
    is_outlier = np.abs(resid_pix - r_mean) > outlier_nsigma * r_std
    print(f"[{tag}]  Total pixels   : {len(resid_pix):,}  (edge_trim={EDGE_TRIM})")
    print(f"[{tag}]  Outlier pixels : {is_outlier.sum():,}  ({100*is_outlier.mean():.2f}%)"
          f"  threshold = ±{outlier_nsigma}σ  (σ={r_std:.4f})")

    w_min, w_max = wave_pix.min(), wave_pix.max()
    bins_w = np.linspace(w_min, w_max, 60)
    bin_centers = 0.5 * (bins_w[:-1] + bins_w[1:])
    bin_idx_arr = np.digitize(wave_pix, bins_w).clip(1, len(bins_w) - 1) - 1

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ax = axes[0]
    rng_o   = np.random.default_rng(42)
    sel_in  = rng_o.choice(np.where(~is_outlier)[0], min(5000, (~is_outlier).sum()), replace=False)
    sel_out = rng_o.choice(np.where( is_outlier)[0], min(2000,  is_outlier.sum()),  replace=False)
    ax.scatter(orig_pix[sel_in],  recon_pix[sel_in],  s=2, alpha=0.2, color='purple',
               label='Normal', rasterized=True)
    ax.scatter(orig_pix[sel_out], recon_pix[sel_out], s=8, alpha=0.6, color='tomato',
               label=f'Outlier (>{outlier_nsigma}σ)', rasterized=True, zorder=5)
    lo, hi = min(orig_pix.min(), recon_pix.min()), max(orig_pix.max(), recon_pix.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
    ax.set_xlabel('Original (norm)'); ax.set_ylabel('Reconstructed (norm)')
    ax.set_title('Orig vs Recon -- outliers highlighted'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1]
    ax.hist(wave_pix,             bins=bins_w, color='steelblue', alpha=0.5, density=True, label='All pixels')
    ax.hist(wave_pix[is_outlier], bins=bins_w, color='tomato',    alpha=0.75, density=True,
            label=f'Outliers (>{outlier_nsigma}σ)')
    _annotate_elines(ax, w_min, w_max)
    ax.set_xlabel('Rest wavelength (µm)'); ax.set_ylabel('Density')
    ax.set_title('Wavelength: outlier pixels vs all'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[2]
    out_rate = np.array([is_outlier[bin_idx_arr == k].mean() if (bin_idx_arr == k).any() else np.nan
                         for k in range(len(bins_w) - 1)])
    ax.bar(bin_centers, out_rate, width=(bins_w[1] - bins_w[0]) * 0.9, color='tomato', alpha=0.8)
    ax.set_ylim(0, np.nanmax(out_rate) * 1.35)
    _annotate_elines(ax, w_min, w_max)
    ax.set_xlabel('Rest wavelength (µm)'); ax.set_ylabel('Outlier fraction')
    ax.set_title('Outlier rate vs wavelength'); ax.grid(alpha=0.3)

    plt.suptitle(f'{tag} -- Outlier Analysis: {is_outlier.sum():,} outlier pixels  '
                 f'({100*is_outlier.mean():.2f}%)   threshold = ±{outlier_nsigma}σ  '
                 f'(edge_trim={EDGE_TRIM})', fontsize=10, fontweight='bold')
    plt.tight_layout(); plt.show()

    # Gallery of spectra with the most outlier pixels.
    offset, ranked = 0, []
    for o, r, w in records:
        n = len(o)
        omask = is_outlier[offset:offset + n]
        ranked.append((omask.sum(), o, r, w, omask)); offset += n
    ranked.sort(key=lambda x: -x[0])
    n_show = min(n_gallery_out, len(ranked))
    n_cols = 4; n_rows = math.ceil(n_show / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows), squeeze=False)
    flat = axes.flatten()
    for ai, (n_out, o, r, w, omask) in enumerate(ranked[:n_show]):
        ax = flat[ai]
        si = np.argsort(w); w_s, o_s, r_s = w[si], o[si], r[si]
        ax.plot(w_s, o_s, color='steelblue', lw=1.2, alpha=0.85, label='Orig')
        ax.plot(w_s, r_s, color='tomato',    lw=1.2, alpha=0.85, ls='--', label='Recon')
        if omask.any():
            ax.scatter(w[omask], o[omask], s=40, color='red', zorder=6, label='Outlier px')
        _annotate_elines(ax, w_s[0], w_s[-1])
        ax.set_title(f'n_outlier={n_out}', fontsize=8)
        ax.set_xlabel('λ rest (µm)', fontsize=7); ax.set_ylabel('Norm flux', fontsize=7)
        ax.tick_params(labelsize=6); ax.grid(alpha=0.25)
        if ai == 0:
            ax.legend(fontsize=6, loc='upper right')
    for ai in range(n_show, len(flat)):
        flat[ai].set_visible(False)
    plt.suptitle(f'{tag} -- spectra with most outlier pixels '
                 '(red dots = outlier pixels, gray dashed = emission lines)',
                 fontsize=10, fontweight='bold')
    plt.tight_layout(); plt.show()


def residual_map(recon_fn, dataset, tag="", outliers_only=False, outlier_nsigma=5.0,
                 z_bins=200, wave_bins=112, batch_size=256,
                 min_sn50=None, min_z=None, max_z=None, cmap=None, metric='mean'):
    """2D map of a reconstruction-residual statistic over (redshift, rest-frame λ).

    Each validation pixel contributes its residual (Recon - Orig, normalised flux)
    at coordinates (object redshift z, REST-frame wavelength λ/(1+z)). Pixels are
    binned on a z x λ_rest grid and reduced per bin by `metric`. Because emission
    lines sit at fixed REST-frame wavelengths, any line-correlated reconstruction
    systematic shows up as a HORIZONTAL stripe; a z- or λ-dependent bias shows up
    as broader vertical/diagonal structure. This is the key diagnostic for whether
    the model mis-reconstructs specific lines or specific spectral regions.

    metric:
        'mean' -> signed mean(Recon-Orig) per bin = bias  (diverging cmap, +/-vmax)
        'mse'  -> mean residual^2 per bin               (sequential cmap, vmin=0)
        'rmse' -> sqrt(MSE) per bin                      (sequential cmap, vmin=0)
    outliers_only: keep only pixels whose residual is > outlier_nsigma sigma from
        the global residual mean, to map WHERE the worst pixels concentrate.
    cmap=None picks a metric-appropriate default (coolwarm / magma); override to force.
    `dataset` is a LowResDataset already built with the desired sn/z cuts; sorting
    its objects by redshift gives a dense, ordered z axis.

    min_sn50 / min_z / max_z: OPTIONAL extra object-level selection applied on top
        of `dataset` (each defaults to None = no cut). When given they sub-select
        objects by dataset.sn50 / dataset.z_best BEFORE sorting, so you can zoom
        the map onto e.g. high-S/N or a redshift slice without rebuilding the
        dataset:  sn50 >= min_sn50,  z_best > min_z,  z_best <= max_z.
    """
    # ---- Optional object-level sub-selection on top of `dataset` (each None = off).
    sn50_all = np.asarray(dataset.sn50, dtype=np.float64)
    z_all    = np.asarray(dataset.z_best, dtype=np.float64)
    sel = np.ones(len(z_all), dtype=bool)
    if min_sn50 is not None:
        sel &= np.isfinite(sn50_all) & (sn50_all >= min_sn50)
    if min_z is not None:
        sel &= np.isfinite(z_all) & (z_all > min_z)
    if max_z is not None:
        sel &= np.isfinite(z_all) & (z_all <= max_z)
    sel_idx = np.where(sel)[0]
    cuts = [c for c in (f"sn50>={min_sn50}" if min_sn50 is not None else None,
                        f"z>{min_z}"        if min_z    is not None else None,
                        f"z<={max_z}"       if max_z    is not None else None) if c]
    print(f"[{tag}]  Selection : {len(sel_idx)}/{len(z_all)} objects"
          f"  [{', '.join(cuts) if cuts else 'no extra cut'}]")

    # ---- Order the SELECTED objects by redshift so the z axis is dense & monotonic.
    order = sel_idx[np.argsort(z_all[sel_idx])]
    z_sorted = z_all[order].astype(np.float32); N = len(order)
    print(f"[{tag}]  Spectra : {N}   z range : [{z_sorted[0]:.3f}, {z_sorted[-1]:.3f}]   edge_trim={EDGE_TRIM}")

    # ---- Reconstruct in batches; collect per-pixel (z, rest-frame λ, residual).
    z_px, wave_px, resid_px = [], [], []
    for start in range(0, N, batch_size):
        sl = slice(start, min(start + batch_size, N))
        idx_b = order[sl]; z_b = z_sorted[sl]
        items   = [dataset[int(i)] for i in idx_b]                 # raw spectra (observed frame)
        flux_bt  = torch.stack([it["flux"] for it in items])
        wave_bt  = torch.stack([it["wavelength"] for it in items])
        vmask_bt = torch.stack([it["valid_mask"] for it in items])
        z_bt     = torch.from_numpy(np.asarray(z_b, dtype=np.float32))
        Rb = recon_fn({"flux": flux_bt.to(DEVICE), "wavelength": wave_bt.to(DEVICE),
                       "valid_mask": vmask_bt.to(DEVICE), "redshift": z_bt})
        w_rest_bt = Rb["wave"] / (1.0 + Rb["redshift"].unsqueeze(-1))  # observed -> rest-frame λ
        L_used = Rb["vmask"].shape[1]
        for bi in range(len(idx_b)):
            # Keep only pixels with a valid-token reconstruction; drop the noisy
            # low-overlap edge pixels (EDGE_TRIM positions per end).
            mi = Rb["recon_mask"][bi].bool().clone().numpy()
            if EDGE_TRIM > 0:
                mi[:EDGE_TRIM] = False; mi[L_used - EDGE_TRIM:] = False
            if mi.sum() < 3:                                       # too few pixels -> skip object
                continue
            w_rest = w_rest_bt[bi][mi].numpy()
            resid  = (Rb["recon_norm"][bi][mi] - Rb["flux_norm"][bi][mi]).numpy()  # Recon - Orig
            si = np.argsort(w_rest)                                # sort by λ (tidy, order-independent)
            z_px.append(np.full(len(w_rest), float(z_b[bi])))     # this object's z, repeated per pixel
            wave_px.append(w_rest[si]); resid_px.append(resid[si])
    z_arr = np.concatenate(z_px); wave_arr = np.concatenate(wave_px); resid_arr = np.concatenate(resid_px)
    print(f"[{tag}]  Total pixels : {len(resid_arr):,}")

    # ---- Optional: restrict to the global residual outliers (where they cluster).
    if outliers_only:
        r_mean, r_std = resid_arr.mean(), resid_arr.std()
        is_out = np.abs(resid_arr - r_mean) > outlier_nsigma * r_std
        print(f"[{tag}]  Outlier pixels : {is_out.sum():,}  ({100*is_out.mean():.2f}%)"
              f"  threshold = +/-{outlier_nsigma}σ  (σ={r_std:.4f})")
        z_arr, wave_arr, resid_arr = z_arr[is_out], wave_arr[is_out], resid_arr[is_out]

    # ---- Bin pixels onto the (z, λ_rest) grid and reduce each bin by `metric`.
    z_edges = np.linspace(z_sorted[0], z_sorted[-1], z_bins + 1)
    wave_edges = np.linspace(wave_arr.min(), wave_arr.max(), wave_bins + 1)
    h_cnt, _, _ = np.histogram2d(z_arr, wave_arr, bins=[z_edges, wave_edges])   # pixels per bin

    # Per-bin statistic = (weighted histogram) / (count). Empty bins -> NaN (blank).
    if metric == 'mean':                       # signed bias -> diverging cmap
        h_sum, _, _ = np.histogram2d(z_arr, wave_arr, bins=[z_edges, wave_edges], weights=resid_arr)
        grid = np.where(h_cnt > 0, h_sum / h_cnt, np.nan)
        metric_label, diverging = 'mean(Recon − Orig)', True
    elif metric in ('mse', 'rmse'):            # magnitude -> sequential cmap, vmin=0
        h_sq, _, _ = np.histogram2d(z_arr, wave_arr, bins=[z_edges, wave_edges], weights=resid_arr ** 2)
        grid = np.where(h_cnt > 0, h_sq / h_cnt, np.nan)          # mean of squares = MSE
        if metric == 'rmse':
            grid = np.sqrt(grid)
        metric_label, diverging = metric.upper(), False
    else:
        raise ValueError(f"metric must be 'mean', 'mse' or 'rmse'; got {metric!r}")

    # ---- Colour scaling: symmetric about 0 for the signed bias, [0, vmax] for
    # magnitudes. vmax = 98th pct to ignore a few extreme bins. cmap default per metric.
    finite = grid[np.isfinite(grid)]
    if diverging:
        vmax = np.nanpercentile(np.abs(finite), 99.5); vmin = -vmax
        this_cmap = cmap if cmap is not None else 'coolwarm'
    else:
        vmax = np.nanpercentile(finite, 98); vmin = 0.0
        this_cmap = cmap if cmap is not None else 'magma'

    # ---- Main panel: the (z, λ_rest) heat-map. Note grid.T -> rows=λ, cols=z.
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [1, 0.04]})
    ax, cax = axes
    pc = ax.pcolormesh(z_edges, wave_edges, grid.T, cmap=this_cmap, vmin=vmin, vmax=vmax, shading='flat')
    fig.colorbar(pc, cax=cax, label=f'{metric_label}  [norm]')

    # ---- Overlay (twin top axis): pixel count per λ bin -> shows where in λ_rest
    # the statistics are well/poorly sampled (sparse bins are noisier).
    wl_counts, wl_edges_l = np.histogram(wave_arr, bins=wave_edges)
    wl_centers = 0.5 * (wl_edges_l[:-1] + wl_edges_l[1:])
    ax_top = ax.twiny()
    ax_top.set_zorder(ax.get_zorder() + 2); ax_top.patch.set_visible(False)
    if not outliers_only:
        ax_top.plot(wl_counts, wl_centers, color='white', lw=3.5, alpha=1.0, zorder=20)   # halo for contrast
        ax_top.plot(wl_counts, wl_centers, color='black', lw=1.6, alpha=0.9, zorder=21,
                    label='pixel λ_rest dist.')
        ax_top.legend(loc='upper right', fontsize=8)
    ax_top.set_xlim(0, wl_counts.max() * 3.0)                     # squeeze curve into right third
    ax_top.set_xlabel(('outlier ' if outliers_only else '') + 'pixel count per λ bin', fontsize=9)
    ax_top.tick_params(axis='x', labelsize=8)

    # ---- Emission-line guides: horizontal lines at fixed rest-frame λ (line
    # systematics in the map align with these), labelled with alternating x offsets.
    w_plot_min, w_plot_max = wave_edges[0], wave_edges[-1]
    for i, (name, wl_rest) in enumerate(EMISSION_LINES.items()):
        if not (w_plot_min <= wl_rest <= w_plot_max):            # only lines inside the λ range
            continue
        ax.axhline(wl_rest, color='k', ls='--', lw=1.1, alpha=0.1)
        x_lbl = z_edges[0] + (z_edges[-1] - z_edges[0]) * (0.97 if i % 2 == 0 else 0.85)
        ax.text(x_lbl, wl_rest, name, ha='right', va='bottom', fontsize=8,
                fontweight='bold', color='#111111',
                bbox=dict(facecolor='white', alpha=0.75, pad=1.5, edgecolor='none'))
    ax.set_xlabel('Redshift  z', fontsize=11)
    ax.set_ylabel('Rest-frame wavelength  (µm)', fontsize=11)
    suffix = ' -- OUTLIER pixels only' if outliers_only else ''
    ax.set_title(f'{tag} -- {metric_label} per (z, λ_rest) bin{suffix}  '
                 f'[n={N} spectra, {z_bins}×{wave_bins} bins, edge_trim={EDGE_TRIM}]\n'
                 'emission lines = horizontal dashed', fontsize=10, fontweight='bold')
    plt.tight_layout(); plt.show()


# Closures binding each protocol to the shared analysis functions.
recon_full = lambda b: reconstruct(model, b, masked=False)
recon_mask = lambda b: reconstruct(model, b, masked=True, block_k=BLOCK_K)

---
## Section A · Full-visible reconstruction

All tokens visible, one forward pass. This is the `val_loss` regime, but note it
is an **untrained regime** — the model never sees fully-unmasked inputs during
training, so reconstructions here can look worse (flatter peaks) than the masked
readout in Section B even for a well-trained model.

In [ ]:
R_full   = reconstruct(model, batch, masked=False)
print(f"recon_flux shape : {R_full['recon_flux'].shape}")
print(f"N tokens         : {R_full['N']}  (P={R_full['P']}, S={R_full['S']}, L_used={R_full['L_used']})")
tok_mse_full = recon_summary(R_full, tag="full-visible")

In [ ]:
gallery(R_full, tag="full-visible",n_gallery=16)
per_token_bar(R_full, tok_mse_full, i=20, tag="full-visible")

In [ ]:
recon_vs_orig(recon_full, tag="full-visible")
mse_vs_wavelength(recon_full, tag="full-visible")

In [ ]:
outlier_analysis(recon_full, tag="full-visible",n_gallery_out=16)

In [ ]:
residual_map(recon_full, dataset=ds, tag="full-visible",
            min_sn50=1, min_z = 0.5, max_z = 4,metric='rmse')
residual_map(recon_full, dataset=ds, tag="full-visible", outliers_only=True, outlier_nsigma=3.0,
            min_sn50=1, min_z = 0.5, max_z = 4,metric='mean')

---
## Section B · Masked (MAE-consistent) reconstruction

For each token a leak-free `BLOCK_K`-token block centred on it is masked and the
model predicts the hidden token from context. This matches the training
objective and is the **trustworthy** reconstruction readout. Costs `N` forward
passes per batch, so it is slower than Section A.

In [ ]:
R_mask   = reconstruct(model, batch, masked=True, block_k=BLOCK_K)
print(f"BLOCK_K = {BLOCK_K}   (1=leave-one-out, ok when stride==patch_size)")
print(f"recon_flux shape : {R_mask['recon_flux'].shape}")
print(f"N tokens         : {R_mask['N']}  (P={R_mask['P']}, S={R_mask['S']}, L_used={R_mask['L_used']})")
tok_mse_mask = recon_summary(R_mask, tag=f"masked K={BLOCK_K}")

In [ ]:
gallery(R_mask, tag=f"masked K={BLOCK_K}")
per_token_bar(R_mask, tok_mse_mask, i=20, tag=f"masked K={BLOCK_K}")

In [ ]:
recon_vs_orig(recon_mask, tag=f"masked K={BLOCK_K}")
mse_vs_wavelength(recon_mask, tag=f"masked K={BLOCK_K}")

In [ ]:
outlier_analysis(recon_mask, tag=f"masked K={BLOCK_K}")

In [ ]:
residual_map(recon_mask, dataset=ds, tag=f"masked K={BLOCK_K}",min_z=0.5, max_z=4, min_sn50=1,metric='rmse')
residual_map(recon_mask, dataset=ds, tag=f"masked K={BLOCK_K}",min_z=0.5, max_z=4, min_sn50=1,metric='mean', outliers_only=True, outlier_nsigma=3.0)

---
## Dataset overview (EDA)

Training-data statistics from the YAML config — independent of reconstruction
protocol, so it is shown once.

In [ ]:
import yaml
from pathlib import Path
from data.dataset import LowResDataset

YAML_PATH = "/home/yacheng/ssl_outthere/encoder_spectrum/LowResPT/low_res_pt.yaml"
with open(YAML_PATH) as f:
    cfg = yaml.safe_load(f)
data_cfg  = cfg["data"]
FITS_PATH = data_cfg["fits_path"]
GRADES    = data_cfg.get("grades", [1, 2, 3])
MIN_OBS   = data_cfg.get("min_obs_frac", 0.5)
MIN_SN50  = data_cfg.get("min_sn50", None)
MIN_Z     = data_cfg.get("min_redshift", None)
WLMIN     = data_cfg.get("wl_ref_min", 1.0)
WLMAX     = data_cfg.get("wl_ref_max", 2.0)
print(f"Config : {YAML_PATH}")
print(f"FITS   : {FITS_PATH}")
print(f"Filters: grades={GRADES}, min_obs_frac={MIN_OBS}, min_sn50={MIN_SN50}, min_redshift={MIN_Z}")

# "All" = grade + obs pre-selection only (the population the model draws from);
# "Kept" additionally applies the sn50 / redshift training cuts.
ds_all = LowResDataset(FITS_PATH, grades=GRADES, min_obs_frac=MIN_OBS,
                       min_sn50=None, min_redshift=None,
                       wl_ref_min=WLMIN, wl_ref_max=WLMAX, use_jansky=True)
sn50   = ds_all.sn50.astype(np.float64)
z_best = ds_all.z_best.astype(np.float64)
flux   = ds_all._flux                              # (N, L) raw f_nu, windowed
valid  = ds_all._valid                             # (N, L) bool, valid_spec flag
N_total = len(sn50)
print(f"\nTotal spectra (grade+obs): {N_total:,}")

mask = np.ones(N_total, dtype=bool)
if MIN_SN50 is not None:
    mask &= np.isfinite(sn50) & (sn50 >= MIN_SN50)
if MIN_Z is not None:
    mask &= np.isfinite(z_best) & (z_best > MIN_Z)
N_kept = mask.sum()
print(f"After sn/z filters       : {N_kept:,}  ({100*N_kept/N_total:.1f}% retained)")

spec_len   = np.full(N_total, flux.shape[1])       # shared 56-px grid (all equal)
valid_pix  = valid.sum(axis=1)
valid_frac = valid_pix / spec_len.clip(min=1)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
flat = axes.flatten()

ax = flat[0]
sn_pos = sn50[np.isfinite(sn50) & (sn50 > 0)]
bins_sn = np.logspace(np.log10(sn_pos.min()), np.log10(sn_pos.max()), 70)
ax.hist(sn_pos, bins=bins_sn, color='steelblue', alpha=0.55, label=f'All ({N_total:,})')
ax.hist(sn50[mask & np.isfinite(sn50) & (sn50 > 0)], bins=bins_sn, color='tomato', alpha=0.75, label=f'Kept ({N_kept:,})')
if MIN_SN50:
    ax.axvline(MIN_SN50, color='k', ls='--', lw=1.5, label=f'cut = {MIN_SN50}')
ax.set_xscale('log'); ax.set_xlabel('SN50 (log scale)'); ax.set_ylabel('Count')
ax.set_title('SN50 Distribution', fontweight='bold'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = flat[1]
z_all = z_best[np.isfinite(z_best)]; z_kept = z_best[mask & np.isfinite(z_best)]
bins_z = np.linspace(z_all.min(), z_all.max(), 60)
ax.hist(z_all,  bins=bins_z, color='steelblue', alpha=0.55, label=f'All ({N_total:,})')
ax.hist(z_kept, bins=bins_z, color='tomato',    alpha=0.75, label=f'Kept ({N_kept:,})')
if MIN_Z is not None:
    ax.axvline(MIN_Z, color='k', ls='--', lw=1.5, label=f'cut = {MIN_Z}')
ax.set_xlabel('Redshift z_best'); ax.set_ylabel('Count')
ax.set_title('Redshift Distribution'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = flat[2]
ax.hist(spec_len,       bins=40, color='steelblue', alpha=0.55, label='All')
ax.hist(spec_len[mask], bins=40, color='tomato',    alpha=0.75, label='Kept')
ax.set_xlabel('Spectrum length (pixels)'); ax.set_ylabel('Count')
ax.set_title('Spectrum Length Distribution'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = flat[3]
ax.hist(valid_frac[mask], bins=50, color='steelblue', alpha=0.8)
med_vf = np.median(valid_frac[mask])
ax.axvline(med_vf, color='tomato', ls='--', lw=1.5, label=f'median = {med_vf:.3f}')
ax.set_xlabel('Valid (non-zero) pixel fraction'); ax.set_ylabel('Count')
ax.set_title('Valid Pixel Fraction  [kept]'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = flat[4]
sel = mask & np.isfinite(sn50) & np.isfinite(z_best) & (sn50 > 0)
ax.scatter(z_best[sel], sn50[sel], s=2, alpha=0.15, color='steelblue', rasterized=True)
ax.set_xlabel('Redshift z_best'); ax.set_ylabel('SN50 (log scale)')
ax.set_yscale('log'); ax.set_title('SN50 vs Redshift  [kept]'); ax.grid(alpha=0.3)

ax = flat[5]; ax.axis('off')
rows = [
    ["N spectra",         f"{N_total:,}", f"{N_kept:,}"],
    ["SN50 median",       f"{np.nanmedian(sn50):.1f}", f"{np.nanmedian(sn50[mask]):.1f}"],
    ["SN50 p25/p75",      f"{np.nanpercentile(sn50,25):.1f} / {np.nanpercentile(sn50,75):.1f}",
                          f"{np.nanpercentile(sn50[mask],25):.1f} / {np.nanpercentile(sn50[mask],75):.1f}"],
    ["z median",          f"{np.nanmedian(z_best):.3f}", f"{np.nanmedian(z_best[mask]):.3f}"],
    ["spec len median",   f"{int(np.median(spec_len))}", f"{int(np.median(spec_len[mask]))}"],
    ["valid frac median", "—", f"{med_vf:.3f}"],
]
tbl = ax.table(cellText=rows, colLabels=["Metric", "All", "Kept"], loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 1.9)
ax.set_title('Summary Statistics', pad=14)

plt.suptitle(f'Dataset EDA — {Path(FITS_PATH).name}\n[min_sn50={MIN_SN50}, min_redshift={MIN_Z}]',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# POSTER FIGURE 1/2 — reconstruction spectra (2x2 packed panels)
#   Original (grey solid) vs Reconstructed (blue dashed) + emission-line markers.
#   Sample #idx and redshift z written inside each panel.
#   Proportions / font sizes matched to 02_linear_probe_st3 poster cells.
#   Transparent background (figure + panels).  Uses R_mask (Section B).
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

POSTER_DIR = "/home/yacheng/ssl_outthere/poster_figs"
os.makedirs(POSTER_DIR, exist_ok=True)

# ---- EDIT ME: which objects go in the 2x2 panels (indices into R_mask batch) ----
id1 = 146
id2 = 186
id3 = 184
id4 = 163
panel_ids = [id1, id2, id3, id4]

R = R_mask
ORIG_C  = '#808080'   # grey -> Original (solid)
RECON_C = '#006bff'   # blue -> Reconstructed (dashed)

fig = plt.figure(figsize=(10, 8))                        # 2x2 of ~4x4 panels (cf. probes 8x4)
gl  = gridspec.GridSpec(2, 2, figure=fig, wspace=0.0, hspace=0.0)
axes = [fig.add_subplot(gl[0, 0]), fig.add_subplot(gl[0, 1]),
        fig.add_subplot(gl[1, 0]), fig.add_subplot(gl[1, 1])]

for k, (ax, idx) in enumerate(zip(axes, panel_ids)):
    m = R["recon_mask"][idx].clone()
    L_used = m.shape[0]
    if EDGE_TRIM > 0:
        m[:EDGE_TRIM] = False
        m[L_used - EDGE_TRIM:] = False
    z  = R["redshift"][idx].item()
    w  = R["wave"][idx][m].numpy()                  # observed frame (shared x grid)
    si = np.argsort(w); w = w[si]
    orig  = R["flux_norm"][idx][m].numpy()[si]
    recon = R["recon_norm"][idx][m].numpy()[si]

    ax.plot(w, orig,  color=ORIG_C,  lw=2.8, alpha=0.95, label='Original')
    ax.plot(w, recon, color=RECON_C, lw=2.8, alpha=0.95, ls='--', label='Reconstructed')
    # emission lines drawn in the OBSERVED frame: rest wavelength * (1+z)
    elo, ehi = w[0], w[-1]
    evis = sorted([(nm, wl*(1.0+z)) for nm, wl in EMISSION_LINES.items()
                   if elo <= wl*(1.0+z) <= ehi], key=lambda t: t[1])
    eylo, eyhi = ax.get_ylim(); eysp = eyhi - eylo
    eylv = [eyhi - 0.04*eysp, eyhi - 0.20*eysp]
    for ei, (enm, ewl) in enumerate(evis):
        ax.axvline(ewl, color='#999999', ls='--', lw=1.0, alpha=0.8)
        ax.text(ewl, eylv[ei % 2], enm, ha='center', va='top', fontsize=10,
                color='#333333', fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.75, pad=1.5, edgecolor='none'))
    ax.text(0.5, 0.06, f'#{idx}   z = {z:.3f}', transform=ax.transAxes,
            ha='center', va='bottom', fontsize=11, fontweight='bold', color=RECON_C,
            bbox=dict(facecolor='white', alpha=0.65, pad=2.0, edgecolor='none'))
    ax.grid(alpha=0.25)
    ax.patch.set_alpha(0.0)
    for sp in ax.spines.values():
        sp.set_linewidth(1.8)
    ax.tick_params(width=1.6, labelsize=11)

    row, col = k // 2, k % 2
    if row == 1:
        ax.set_xlabel(r'$\lambda_{\rm obs}$  (µm)', fontsize=13, fontweight='semibold')
    else:
        ax.tick_params(labelbottom=False)
    if col == 0:
        ax.set_ylabel('Norm. flux', fontsize=13, fontweight='semibold')
    else:
        ax.tick_params(labelleft=False)
    if k == 0:
        leg = ax.legend(fontsize=11, loc='upper right', framealpha=0.8)
        leg.get_frame().set_linewidth(2.0)
        leg.get_frame().set_edgecolor('black')

fig.patch.set_alpha(0.0)
outp = os.path.join(POSTER_DIR, 'poster_recon_spectra')
fig.savefig(outp + '.png', transparent=True, dpi=200, bbox_inches='tight')
print(f'Saved → {outp}.png')
plt.show()


In [ ]:
# ============================================================
# POSTER FIGURE 2/2 — Original vs Reconstructed 2D histogram (all val pixels)
#   viridis cmap, black 1:1 line; empty bins transparent.
#   Proportions / font sizes matched to 02_linear_probe_st3 poster cells.
# ============================================================
import os, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker

POSTER_DIR = "/home/yacheng/ssl_outthere/poster_figs"
os.makedirs(POSTER_DIR, exist_ok=True)

HIST_CMAP = 'YlGnBu_r'          # blue->green->yellow (alts: 'GnBu', 'YlGnBu', 'winter')
LINE11_C  = 'black'            # 1:1 line (high contrast over viridis)

# pool Original vs Reconstructed pixels over the whole val set (masked protocol)
orig_pool, recon_pool = [], []
for b in dm.val_dataloader():
    Rb = reconstruct(model, b, masked=True, block_k=BLOCK_K)
    m  = Rb["recon_mask"].bool().clone()
    L_used = m.shape[1]
    if EDGE_TRIM > 0:
        m[:, :EDGE_TRIM] = False
        m[:, L_used - EDGE_TRIM:] = False
    orig_pool.append(Rb["flux_norm"][m].numpy())
    recon_pool.append(Rb["recon_norm"][m].numpy())
orig_all  = np.concatenate(orig_pool)
recon_all = np.concatenate(recon_pool)
r_val     = np.corrcoef(orig_all, recon_all)[0, 1]
r2        = r_val ** 2

PCT = (0.1, 99.7)
lo  = min(np.percentile(orig_all, PCT[0]), np.percentile(recon_all, PCT[0]))
hi  = max(np.percentile(orig_all, PCT[1]), np.percentile(recon_all, PCT[1]))
pad = 0.08 * (hi - lo)
lo -= pad; hi += 0.3 * pad
edges = np.linspace(lo, hi, 130)
H, xe, ye = np.histogram2d(orig_all, recon_all, bins=[edges, edges])
Hm = np.ma.masked_where(H.T == 0, H.T)

fig, axes = plt.subplots(1, 1, figsize=(7, 7))
ax = axes
cmap = plt.get_cmap(HIST_CMAP).copy()
cmap.set_bad(alpha=0.0)
vmax = 10 ** math.ceil(math.log10(float(Hm.max())))     # round up so the top decade tick shows
pcm = ax.pcolormesh(xe, ye, Hm, cmap=cmap, norm=mcolors.LogNorm(vmin=1, vmax=vmax))
cb  = fig.colorbar(pcm, ax=ax, orientation='horizontal', fraction=0.05, pad=0.18)
cb.set_label('pixel count', fontsize=13, fontweight='semibold')
cb.ax.xaxis.set_major_locator(mticker.LogLocator(base=10))
cb.ax.xaxis.set_minor_locator(mticker.LogLocator(base=10, subs=np.arange(2, 10)))
cb.ax.xaxis.set_major_formatter(mticker.LogFormatterMathtext(base=10))
cb.ax.tick_params(which='both', labelsize=11, width=1.6)
for t in cb.ax.get_xticklabels():
    t.set_fontweight('semibold')
cb.outline.set_linewidth(1.8)

ax.plot([lo, hi], [lo, hi], ls='--', lw=1.5, color=LINE11_C, label='1:1')
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')
ax.set_xlabel('Original  (norm. flux)', fontsize=13, fontweight='semibold')
ax.set_ylabel('Reconstructed  (norm. flux)', fontsize=13, fontweight='semibold')
ax.set_title(f'$R^2$ = {r2:.3f}    N = {len(orig_all):,} px', fontsize=15, fontweight='semibold')
leg = ax.legend(fontsize=11, loc='upper left', framealpha=0.8)
leg.get_frame().set_linewidth(2.0)
leg.get_frame().set_edgecolor('black')
ax.grid(alpha=0.25)
ax.patch.set_alpha(0.0)
for sp in ax.spines.values():
    sp.set_linewidth(1.8)
ax.tick_params(width=1.6, labelsize=11)

fig.patch.set_alpha(0.0)
plt.tight_layout()
outp = os.path.join(POSTER_DIR, 'poster_recon_hist2d')
fig.savefig(outp + '.png', transparent=True, dpi=200)
print(f'Saved → {outp}.png')
plt.show()
